# NBA Quant AI — GPU Evolution v2 (Optimized)

**10x faster than v1** — TabICL-focused, Google Drive persistence, 2-fold fast eval.

| Optimization | v1 | v2 | Speedup |
|---|---|---|---|
| Population | 80 | 24 | 3.3x |
| CV Folds | 5 | 2 | 2.5x |
| Models | 7 (trees+GPU) | TabICL only | 1x (focused) |
| Feature cache | Rebuild each session | Google Drive | Save 36 min |
| State save | Every 25 gens | Every gen to Drive | Survives crash |
| Eval timeout | None | 90s | No hanging |
| Data subsample | All 9551 games | Last 6000 | 1.6x |

**Expected: ~50-70 gens/hour on free T4** (vs 3 gens/hour in v1)

**BEFORE RUNNING**: Runtime → GPU (T4) → Secrets: `HF_TOKEN`, `DATABASE_URL`

In [ ]:
# ============================================================
# CELL 1: SETUP — CUDA + DEPS + DRIVE (run once per session)
# ============================================================
import subprocess, sys, os, time, gc, json, warnings
warnings.filterwarnings('ignore')  # Kill sklearn spam

# Mount Google Drive for persistent cache
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE_DIR = '/content/drive/MyDrive/nba-quant-gpu'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive dir: {DRIVE_DIR}')

# Secrets
from google.colab import userdata
for key in ['HF_TOKEN', 'DATABASE_URL']:
    try:
        v = userdata.get(key)
        if v: os.environ[key] = v; print(f'{key}: OK')
    except: print(f'{key}: not set')

# Install deps (minimal — no torchvision, no tabpfn which is broken)
t0 = time.time()
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'torch --index-url https://download.pytorch.org/whl/cu121'.split()[0],
    '--index-url', 'https://download.pytorch.org/whl/cu121'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'xgboost', 'lightgbm', 'catboost', 'scikit-learn', 'pandas', 'numpy',
    'scipy', 'tabicl', 'huggingface_hub', 'psycopg2-binary', 'requests'])
print(f'Deps installed: {time.time()-t0:.0f}s')

# Verify CUDA
import torch, numpy as np
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB)')

# Pre-cache TabICL checkpoint
t0 = time.time()
try:
    from tabicl import TabICLClassifier
    _m = TabICLClassifier(); _m.fit(np.random.randn(50,5), np.random.randint(0,2,50)); del _m
    print(f'TabICL checkpoint: cached ({time.time()-t0:.0f}s)')
except Exception as e:
    print(f'TabICL: FAILED ({e})')
gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# ============================================================
# CELL 2: BUILD OR LOAD FEATURES (cached on Google Drive)
# ============================================================
from pathlib import Path

CACHE_FILE = Path(f'{DRIVE_DIR}/features_cache_v38.npz')  # versioned by engine version

if CACHE_FILE.exists():
    print('Loading cached features from Google Drive...')
    t0 = time.time()
    cached = np.load(str(CACHE_FILE), allow_pickle=True)
    X_full = cached['X']
    y_full = cached['y']
    feature_names = cached['feature_names'].tolist()
    print(f'Loaded: {X_full.shape} in {time.time()-t0:.1f}s')
else:
    print('Building features (~30 min, will cache to Drive)...')
    t0 = time.time()
    # Clone repo
    if not os.path.exists('/content/nomos-nba-agent'):
        subprocess.run(['git', 'clone', '--depth=1',
                        'https://github.com/LBJLincoln/nomos-nba-agent.git',
                        '/content/nomos-nba-agent'], check=True)
    sys.path.insert(0, '/content/nomos-nba-agent/hf-space')

    data_dir = Path('/content/nomos-nba-agent/hf-space/data/historical')
    games = []
    for f in sorted(data_dir.glob('games-*.json')):
        raw = json.loads(f.read_text())
        if isinstance(raw, list): games.extend(raw)
        elif isinstance(raw, dict) and 'games' in raw: games.extend(raw['games'])
    print(f'Games: {len(games)}')

    from features.engine import NBAFeatureEngine
    engine = NBAFeatureEngine()
    X_raw, y_raw, feature_names = engine.build(games)
    X_full = np.nan_to_num(np.array(X_raw, dtype=np.float32))  # float32 saves RAM
    y_full = np.array(y_raw, dtype=np.int32)
    variances = np.var(X_full, axis=0)
    valid = variances > 1e-10
    X_full = X_full[:, valid]
    feature_names = [f for f, v in zip(feature_names, valid) if v]
    np.savez_compressed(str(CACHE_FILE), X=X_full, y=y_full,
                        feature_names=np.array(feature_names))
    print(f'Built & cached to Drive: {X_full.shape} in {time.time()-t0:.0f}s')

# Subsample: last 6000 games (faster eval, more recent = more relevant)
SUBSAMPLE = 6000
if X_full.shape[0] > SUBSAMPLE:
    X = X_full[-SUBSAMPLE:]
    y = y_full[-SUBSAMPLE:]
    print(f'Subsampled: {X_full.shape[0]} → {SUBSAMPLE} (last {SUBSAMPLE} games)')
else:
    X = X_full; y = y_full

n_features = X.shape[1]
print(f'Ready: {X.shape} ({n_features} features)')

In [ ]:
# ============================================================
# CELL 3: SEED POPULATION FROM 6 ISLANDS + CONFIG
# ============================================================
import requests, random, signal
from collections import Counter
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import brier_score_loss
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.calibration import CalibratedClassifierCV
import xgboost as xgb
import lightgbm as lgb

# ── PLATFORM CONFIG ──
# Adjust these based on your platform:
#   Colab Free T4:  POP=24, FOLDS=2, GENS=500
#   Colab Pro A100:  POP=40, FOLDS=3, GENS=500
#   Lightning.ai:    POP=40, FOLDS=3, GENS=1000
PLATFORM = 'colab_free'  # 'colab_free', 'colab_pro', 'lightning'

CONFIGS = {
    'colab_free':  {'POP': 24, 'FOLDS': 2, 'GENS': 500, 'ELITE': 4, 'TIMEOUT': 90},
    'colab_pro':   {'POP': 40, 'FOLDS': 3, 'GENS': 500, 'ELITE': 6, 'TIMEOUT': 120},
    'lightning':   {'POP': 40, 'FOLDS': 3, 'GENS': 1000, 'ELITE': 6, 'TIMEOUT': 120},
}
cfg = CONFIGS[PLATFORM]
POP_SIZE = cfg['POP']
N_SPLITS = cfg['FOLDS']
TOTAL_GENS = cfg['GENS']
ELITE_SIZE = cfg['ELITE']
EVAL_TIMEOUT = cfg['TIMEOUT']

# Evolution params
PURGE_GAP = 5
TARGET_FEATURES = 60
MAX_FEATURES = 200
MUTATION_RATE = 0.10
CROSSOVER_RATE = 0.80
MUT_FLOOR = 0.05
MUT_DECAY = 0.998

# GPU-capable model types (TabICL is the star, trees for diversity)
_xgb_device = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU_MODEL_TYPES = ['tabicl', 'xgboost', 'catboost', 'lightgbm', 'extra_trees']
# Weight distribution: 50% TabICL, 50% trees (trees provide feature selection pressure)
MODEL_WEIGHTS = {'tabicl': 50, 'xgboost': 15, 'catboost': 10, 'lightgbm': 10, 'extra_trees': 15}

# Walk-forward splits
tscv = TimeSeriesSplit(n_splits=N_SPLITS)
splits = [(tr[:-PURGE_GAP] if len(tr) > PURGE_GAP + 50 else tr, te) for tr, te in tscv.split(X)]

print(f'Platform: {PLATFORM} | Pop={POP_SIZE} | Folds={N_SPLITS} | Gens={TOTAL_GENS}')
print(f'XGBoost device: {_xgb_device}')
print(f'Estimated: ~{POP_SIZE * 8}s/gen = ~{3600 // (POP_SIZE * 8)} gens/hour')

# ── Fetch seeds from 6 islands (updated URLs) ──
ISLANDS = {
    'S10': 'https://nomos42-nba-quant.hf.space',
    'S11': 'https://nomos42-nba-quant-2.hf.space',
    'S12': 'https://nomos42-nba-evo-3.hf.space',
    'S13': 'https://nomos42-nba-evo-4.hf.space',
    'S14': 'https://nomos42-nba-evo-5.hf.space',
    'S15': 'https://nomos42-nba-evo-6.hf.space',
}

island_seeds = []
print('\nFetching seeds from islands...')
for name, url in ISLANDS.items():
    try:
        resp = requests.get(f'{url}/api/results', timeout=10)
        if resp.status_code != 200: continue
        data = resp.json()
        best = data.get('best', {})
        island_seeds.append({
            'source': name, 'brier': best.get('brier', 1.0),
            'features': best.get('selected_features', []),
            'model_type': best.get('model_type', 'xgboost'),
        })
        print(f'  {name}: brier={best.get("brier","?"):.5f} model={best.get("model_type","?")} feat={best.get("n_features","?")}')
        for i, ind in enumerate(data.get('top5', [])[:2]):
            island_seeds.append({
                'source': f'{name}_top{i+1}', 'brier': ind.get('brier', 1.0),
                'features': ind.get('selected_features', []),
                'model_type': ind.get('model_type', 'xgboost'),
            })
    except Exception as e:
        print(f'  {name}: {e}')

print(f'Seeds collected: {len(island_seeds)}')

In [ ]:
# ============================================================
# CELL 4: EVOLUTION ENGINE (optimized)
# ============================================================

def make_model(model_type, hp):
    if model_type == 'xgboost':
        return xgb.XGBClassifier(
            n_estimators=min(hp.get('n_estimators', 200), 300),
            max_depth=hp.get('max_depth', 6),
            learning_rate=hp.get('learning_rate', 0.05),
            subsample=hp.get('subsample', 0.8),
            colsample_bytree=hp.get('colsample_bytree', 0.8),
            tree_method='hist', device=_xgb_device,
            random_state=42, verbosity=0)
    elif model_type == 'lightgbm':
        return lgb.LGBMClassifier(
            n_estimators=min(hp.get('n_estimators', 200), 300),
            max_depth=hp.get('max_depth', 6),
            learning_rate=hp.get('learning_rate', 0.05),
            subsample=hp.get('subsample', 0.8),
            colsample_bytree=hp.get('colsample_bytree', 0.8),
            random_state=42, verbose=-1)
    elif model_type == 'extra_trees':
        return ExtraTreesClassifier(
            n_estimators=min(hp.get('n_estimators', 200), 300),
            max_depth=hp.get('max_depth', 8),
            min_samples_leaf=5, random_state=42, n_jobs=-1)
    elif model_type == 'catboost':
        from catboost import CatBoostClassifier
        return CatBoostClassifier(
            iterations=min(hp.get('n_estimators', 200), 300),
            depth=min(hp.get('max_depth', 6), 10),
            learning_rate=hp.get('learning_rate', 0.05),
            random_state=42, verbose=0)
    elif model_type == 'tabicl':
        from tabicl import TabICLClassifier
        return TabICLClassifier()
    else:
        return ExtraTreesClassifier(n_estimators=200, random_state=42, n_jobs=-1)

class _Timeout(Exception): pass
def _timeout_handler(signum, frame): raise _Timeout()

def evaluate(features_mask, model_type, hp, timeout=EVAL_TIMEOUT):
    """Evaluate with timeout protection."""
    indices = [i for i, v in enumerate(features_mask) if v]
    if len(indices) < 5: return 1.0
    if len(indices) > MAX_FEATURES: indices = indices[:MAX_FEATURES]
    X_sub = X[:, indices].astype(np.float32)

    # Set timeout
    old_handler = signal.signal(signal.SIGALRM, _timeout_handler)
    signal.alarm(timeout)

    try:
        fold_briers = []
        for train_idx, test_idx in splits:
            model = make_model(model_type, hp)
            model.fit(X_sub[train_idx], y[train_idx])
            probs = model.predict_proba(X_sub[test_idx])[:, 1]
            fold_briers.append(brier_score_loss(y[test_idx], probs))
            del model
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        return float(np.mean(fold_briers))
    except _Timeout:
        signal.signal(signal.SIGALRM, old_handler)
        gc.collect()
        return 0.30  # penalty for timeout
    except Exception:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)
        return 0.30

class Individual:
    def __init__(self, features=None, model_type=None, hp=None):
        if features is None:
            prob = TARGET_FEATURES / max(n_features, 1)
            self.features = [1 if random.random() < prob else 0 for _ in range(n_features)]
        else:
            self.features = list(features)
        self.model_type = model_type or random.choices(
            list(MODEL_WEIGHTS.keys()), weights=list(MODEL_WEIGHTS.values()), k=1)[0]
        self.hp = hp or {
            'n_estimators': random.randint(100, 300),
            'max_depth': random.randint(4, 9),
            'learning_rate': 10 ** random.uniform(-2.0, -0.7),
            'subsample': random.uniform(0.6, 1.0),
            'colsample_bytree': random.uniform(0.4, 1.0),
        }
        self.brier = 1.0
        self.n_feat = sum(self.features)
        self._enforce_cap()

    def _enforce_cap(self):
        indices = [i for i, v in enumerate(self.features) if v]
        if len(indices) > MAX_FEATURES:
            for i in random.sample(indices, len(indices) - MAX_FEATURES):
                self.features[i] = 0
        self.n_feat = sum(self.features)

    def eval(self):
        self.brier = evaluate(self.features, self.model_type, self.hp)
        return self.brier

    def mutate(self, rate):
        for i in range(len(self.features)):
            if random.random() < rate:
                self.features[i] = 1 - self.features[i]
        if random.random() < 0.25: self.hp['n_estimators'] = max(50, min(300, self.hp['n_estimators'] + random.randint(-50, 50)))
        if random.random() < 0.25: self.hp['max_depth'] = max(2, min(10, self.hp['max_depth'] + random.randint(-2, 2)))
        if random.random() < 0.25: self.hp['learning_rate'] = max(0.001, min(0.3, self.hp['learning_rate'] * 10 ** random.uniform(-0.3, 0.3)))
        if random.random() < 0.08:
            self.model_type = random.choices(
                list(MODEL_WEIGHTS.keys()), weights=list(MODEL_WEIGHTS.values()), k=1)[0]
        self._enforce_cap()
        self.brier = 1.0

    @staticmethod
    def crossover(p1, p2):
        point = random.randint(1, len(p1.features) - 1)
        child = Individual(
            features=p1.features[:point] + p2.features[point:],
            model_type=p1.model_type if random.random() < 0.5 else p2.model_type,
            hp={k: p1.hp[k] if random.random() < 0.5 else p2.hp[k] for k in p1.hp})
        return child

# ── Build population ──
name_to_idx = {name: i for i, name in enumerate(feature_names)}

# Load state from Drive if exists (resume from previous session)
STATE_FILE = Path(f'{DRIVE_DIR}/evolution_state_v2.json')
population = []
best_ever_brier = 1.0
best_ever_info = None
start_gen = 0

if STATE_FILE.exists():
    try:
        state = json.loads(STATE_FILE.read_text())
        start_gen = state.get('generation', 0)
        best_ever_brier = state.get('best_brier', 1.0)
        best_ever_info = state.get('best_info')
        # Restore population from saved individuals
        for saved in state.get('population', []):
            feat = [0] * n_features
            for fname in saved.get('features', []):
                if fname in name_to_idx: feat[name_to_idx[fname]] = 1
            ind = Individual(features=feat, model_type=saved.get('model_type', 'tabicl'),
                           hp=saved.get('hp', {}))
            ind.brier = saved.get('brier', 1.0)
            population.append(ind)
        print(f'RESUMED: gen={start_gen}, best={best_ever_brier:.5f}, pop={len(population)}')
    except Exception as e:
        print(f'State load failed: {e}')

# Seed from islands if no state
if len(population) < POP_SIZE:
    for seed in island_seeds:
        if len(population) >= POP_SIZE: break
        feat = [0] * n_features
        for fname in seed.get('features', []):
            if isinstance(fname, str) and fname in name_to_idx:
                feat[name_to_idx[fname]] = 1
        if sum(feat) < 15:
            prob = TARGET_FEATURES / max(n_features, 1)
            feat = [1 if random.random() < prob else 0 for _ in range(n_features)]
        # Create TabICL variant of every seed (the whole point of GPU evolution)
        population.append(Individual(features=feat, model_type='tabicl'))
        if len(population) < POP_SIZE:
            population.append(Individual(features=list(feat), model_type=seed.get('model_type', 'xgboost')))

    while len(population) < POP_SIZE:
        population.append(Individual())
    population = population[:POP_SIZE]

mt_counts = Counter(ind.model_type for ind in population)
print(f'Population: {len(population)} | Models: {dict(mt_counts)}')

In [ ]:
# ============================================================
# CELL 5: EVOLUTION LOOP (with Drive checkpointing)
# ============================================================
print('='*70)
print(f'  NBA QUANT AI — GPU EVOLUTION v2 ({PLATFORM})')
print(f'  Pop={POP_SIZE} | Folds={N_SPLITS} | Gens={TOTAL_GENS} | Resume={start_gen}')
print(f'  ATR to beat: 0.21837 (S13 CatBoost gen815)')
print('='*70)

mut_rate = MUTATION_RATE
session_start = time.time()
gens_this_session = 0

def save_state(gen, population, best_brier, best_info, mut_rate):
    """Save full state to Google Drive — survives Colab disconnects."""
    pop_data = []
    for ind in population:
        pop_data.append({
            'features': [feature_names[i] for i, v in enumerate(ind.features) if v],
            'model_type': ind.model_type,
            'hp': ind.hp,
            'brier': ind.brier,
        })
    STATE_FILE.write_text(json.dumps({
        'generation': gen + 1,
        'best_brier': best_brier,
        'best_info': best_info,
        'mutation_rate': mut_rate,
        'population': pop_data,
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    }, default=str))

for gen in range(start_gen, TOTAL_GENS):
    gen_start = time.time()

    # Evaluate unevaluated
    n_eval = 0
    for ind in population:
        if ind.brier >= 0.99:
            ind.eval()
            n_eval += 1

    population.sort(key=lambda x: x.brier)

    # Track best
    gen_best = population[0]
    improved = gen_best.brier < best_ever_brier
    if improved:
        best_ever_brier = gen_best.brier
        best_ever_info = {
            'brier': gen_best.brier, 'model_type': gen_best.model_type,
            'n_features': gen_best.n_feat, 'generation': gen + 1,
            'features': [feature_names[i] for i, v in enumerate(gen_best.features) if v],
            'hp': gen_best.hp,
        }

    gen_dur = time.time() - gen_start
    elapsed = (time.time() - session_start) / 60
    gens_this_session += 1
    rate = gens_this_session / max(elapsed, 0.1) * 60  # gens/hour

    marker = ' *** NEW BEST ***' if improved else ''
    print(f'Gen {gen+1}/{TOTAL_GENS}: best={gen_best.brier:.5f} ({gen_best.model_type}, {gen_best.n_feat}f) '
          f'| ever={best_ever_brier:.5f} | {n_eval} evals {gen_dur:.0f}s | '
          f'{elapsed:.0f}min {rate:.0f}g/h{marker}')

    # Detailed log every 10 gens
    if (gen + 1) % 10 == 0:
        mt = Counter(ind.model_type for ind in population)
        top5 = [(f'{ind.brier:.5f}', ind.model_type, ind.n_feat) for ind in population[:5]]
        print(f'  Models: {dict(mt)} | Top5: {top5}')

    # Save state to Drive EVERY gen (cheap, survives crashes)
    save_state(gen, population, best_ever_brier, best_ever_info, mut_rate)

    # ── Selection + Reproduction ──
    elite = population[:ELITE_SIZE]
    def tournament(pop, k=4):
        return min(random.sample(pop, min(k, len(pop))), key=lambda x: x.brier)

    children = []
    for e in elite:
        c = Individual(features=list(e.features), model_type=e.model_type, hp=dict(e.hp))
        c.brier = e.brier
        children.append(c)

    while len(children) < POP_SIZE:
        if random.random() < CROSSOVER_RATE:
            child = Individual.crossover(tournament(population), tournament(population))
        else:
            p = tournament(population)
            child = Individual(features=list(p.features), model_type=p.model_type, hp=dict(p.hp))
        child.mutate(mut_rate)

        # Force TabICL representation (50% of new children)
        if random.random() < 0.40:
            child.model_type = 'tabicl'
        children.append(child)

    population = children[:POP_SIZE]
    mut_rate = max(MUT_FLOOR, min(0.15, mut_rate * MUT_DECAY))

# Final save
save_state(TOTAL_GENS - 1, population, best_ever_brier, best_ever_info, mut_rate)
print(f'\nDone! {gens_this_session} gens in {(time.time()-session_start)/60:.0f} min')

In [ ]:
# ============================================================
# CELL 6: RESULTS + INJECT BACK INTO HF ISLANDS
# ============================================================
print('\n' + '='*70)
print('  FINAL RESULTS')
print('='*70)

if best_ever_info:
    print(f'  Best Brier:    {best_ever_info["brier"]:.5f}')
    print(f'  Model:         {best_ever_info["model_type"]}')
    print(f'  Features:      {best_ever_info["n_features"]}')
    print(f'  Generation:    {best_ever_info["generation"]}')
    print(f'  Session gens:  {gens_this_session}')
    print(f'  Session time:  {(time.time()-session_start)/60:.0f} min')

    population.sort(key=lambda x: x.brier)
    print(f'\n  Top 10:')
    for i, ind in enumerate(population[:10]):
        print(f'    #{i+1}: brier={ind.brier:.5f} | {ind.model_type} | {ind.n_feat}f')

    mt_top = Counter(ind.model_type for ind in population[:20])
    print(f'\n  Models in top 20: {dict(mt_top)}')

    # Compare to ATR
    print(f'\n  Current ATR:  0.21837 (S13 CatBoost)')
    delta = best_ever_info['brier'] - 0.21837
    print(f'  Delta vs ATR: {delta:+.5f} {"NEW RECORD!" if delta < 0 else ""}')

    # Save best features for injection into HF islands
    best_features_file = Path(f'{DRIVE_DIR}/best_gpu_features.json')
    best_features_file.write_text(json.dumps({
        'brier': best_ever_info['brier'],
        'model_type': best_ever_info['model_type'],
        'features': best_ever_info['features'],
        'n_features': best_ever_info['n_features'],
        'generation': best_ever_info['generation'],
        'source': f'gpu_v2_{PLATFORM}',
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    }, indent=2))
    print(f'\n  Best features saved to: {best_features_file}')
    print(f'  → Use these features as seeds in HF CPU islands')
    print(f'  → Inject via POST /api/config on S10-S15')
else:
    print('  No results yet.')